# Stop words and hashing

- The next steps will be to remove stop words and then apply the hashing trick, converting the results into a TF-IDF.

A quick reminder about these concepts:

- The hashing trick provides a fast and space-efficient way to map a very large (possibly infinite) set of items (in this case, all words contained in the SMS messages) onto a smaller, finite number of values.
- The TF-IDF matrix reflects how important a word is to each document. It takes into account both the frequency of the word within each document but also the frequency of the word across all of the documents in the collection.

The tokenized SMS data are stored in `sms` in a column named `words`. You've cleaned up the handling of spaces in the data so that the tokenized text is neater.

## Instructions

- Import the `StopWordsRemover`, `HashingTF` and `IDF` classes.
- Create a `StopWordsRemover` object (input column `words`, output column `terms`). Apply to `sms`.
- Create a `HashingTF` object (input results from previous step, output column `hash`). Apply to `wrangled`.
- Create an `IDF` object (input results from previous step, output column `features`). Apply to `wrangled`.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('sms_manipulate_columns').getOrCreate()


In [ ]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [ ]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType()),
	StructField("text", StringType()),
	StructField("label", IntegerType())
])
	
sms = spark.read.csv("file:///home/talentum/test-jupyter/c5-MLWithPySpark/M2-Classification/4_TurningTextIntoTables/dataset/sms.csv", sep=';', header=False, schema=schema)

print("First few rows from the sms DataFrame:")
sms.show(4, truncate=False)

First few rows from the sms DataFrame:
+---+-------------------------------------------+-----+
|id |text                                       |label|
+---+-------------------------------------------+-----+
|1  |Sorry, I'll call later in meeting          |0    |
|2  |Dont worry. I guess he's busy.             |0    |
|3  |Call FREEPHONE 0800 542 0578 now!          |1    |
|4  |Win a 1000 cash prize or a prize worth 5000|1    |
+---+-------------------------------------------+-----+
only showing top 4 rows



In [8]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType()),
	StructField("text", StringType()),
	StructField("label", IntegerType())
])
	
sms = spark.read.csv("file:///home/talentum/test-jupyter/c5-MLWithPySpark/M2-Classification/4_TurningTextIntoTables/dataset/sms.csv", sep=';', header=False, schema=schema)

from pyspark.sql.functions import regexp_replace
from pyspark.ml.feature import Tokenizer

sms = sms.withColumn('text', regexp_replace(sms.text, '[_():;,.!?\\\\\\\\-]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '[0-9]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '(^ +| +$)', ''))
sms = sms.withColumn('text', regexp_replace(sms.text, ' +', ' '))
sms = Tokenizer(inputCol='text', outputCol='words').transform(sms)
sms = sms.select('id', 'words', 'label')

print("First few rows from the sms DataFrame:")

+---+----------------------------------+-----+------------------------------------------+
|id |text                              |label|words                                     |
+---+----------------------------------+-----+------------------------------------------+
|1  |Sorry I'll call later in meeting  |0    |[sorry, i'll, call, later, in, meeting]   |
|2  |Dont worry I guess he's busy      |0    |[dont, worry, i, guess, he's, busy]       |
|3  |Call FREEPHONE now                |1    |[call, freephone, now]                    |
|4  |Win a cash prize or a prize worth |1    |[win, a, cash, prize, or, a, prize, worth]|
+---+----------------------------------+-----+------------------------------------------+
only showing top 4 rows



In [ ]:
from pyspark.ml.____ import ____, ____, ____

# Remove stop words.
wrangled = ____(inputCol=____, outputCol=____)\
      .____(sms)

# Apply the hashing trick
wrangled = ____(____, ____, numFeatures=1024)\
      .____(wrangled)

# Convert hashed symbols to TF-IDF
tf_idf = ____(____, ____)\
      .____(wrangled).____(wrangled)
      
tf_idf.select('terms', 'features').show(4, truncate=False)